<a href="https://colab.research.google.com/github/Jiyaapatil35/Rag_Powered_Financial_Fraud_Detection/blob/main/GenAI_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Financial Fraud Documentation and Explanation Using RAG and LLMs**

## Library Installation and Setup

This cell installs all the required libraries for the Financial Fraud Detection System. These packages are used for PDF extraction, OCR-based image reading, vector database creation for the RAG pipeline, running LLM models like Phi-2, and building the Gradio web interface. The setup prepares the Google Colab environment for the complete fraud analysis workflow.

In [ ]:
!pip install pdfplumber pytesseract pillow
!apt-get install tesseract-ocr -y

!pip install sentence-transformers faiss-cpu
!pip install transformers accelerate bitsandbytes
!pip install gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 54.2 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00


## Importing Required Libraries

This cell imports all the necessary libraries used in the Financial Fraud Detection System. These libraries support PDF and image text extraction, vector embedding generation, FAISS-based retrieval for the RAG pipeline, LLM integration using transformers, and Gradio-based user interface development.

In [ ]:
import pdfplumber
import pytesseract
from PIL import Image
import re
import json
import numpy as np

from sentence_transformers import SentenceTransformer
import faiss

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

import gradio as gr

## Loading Sentence Transformer Model

This cell loads the `all-MiniLM-L6-v2` sentence transformer model, which is used to convert fraud-related documents and transaction text into vector embeddings. These embeddings help the RAG system perform semantic search and retrieve relevant fraud knowledge from the FAISS vector database.

In [ ]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Defining Fraud Detection Rules

This cell defines a set of predefined fraud detection rules used in the RAG pipeline. These rules represent common suspicious transaction patterns such as unusual transaction amounts, unknown receivers, international transfers, and transactions at odd hours. The rules are later converted into embeddings and stored in the FAISS vector database for semantic retrieval during fraud analysis.

In [ ]:
fraud_rules = [
    "Transaction amount unusually high compared to past transactions",
    "Multiple rapid transactions in short time",
    "Transaction from unusual location",
    "Unknown receiver account",
    "International transfer without history",
    "Mismatch in sender and receiver names",
    "Suspicious keywords like lottery, prize, urgent transfer",
    "Transaction at odd hours",
    "Frequent small transactions (smurfing)",
    "Account used after long inactivity"
]

## Creating Embeddings and FAISS Vector Database

This cell converts the fraud detection rules into vector embeddings using the sentence transformer model. The embeddings are then stored in a FAISS vector database, which enables semantic similarity search and retrieval of relevant fraud rules in the RAG pipeline.

In [ ]:
rule_embeddings = embed_model.encode(fraud_rules)

dimension = rule_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(rule_embeddings))

## Loading the Phi-2 Large Language Model

This cell loads the Phi-2 Large Language Model and its tokenizer using the Hugging Face transformers library. Additional configurations such as `pad_token` and `pad_token_id` are set to avoid tokenization and generation errors during inference. The model is used to analyze transactions, generate fraud explanations, and provide intelligent responses within the RAG-based fraud detection system.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "microsoft/phi-2"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# FIX 1: Ensure pad token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 🔥 IMPORTANT FIX: Load config first and MODIFY it
from transformers import AutoConfig
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

# Manually inject pad_token_id BEFORE model loads
config.pad_token_id = tokenizer.eos_token_id

# Now load model using modified config
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    config=config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## PDF Text Extraction

This cell defines a function to extract text from uploaded PDF bank statements using the `pdfplumber` library. The extracted transaction data is later processed by the RAG and LLM pipeline for financial fraud analysis and explanation.

In [ ]:
def extract_text_from_pdf(file):
    text = ""
    with pdfplumber.open(file) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text

## Image Text Extraction using OCR

This cell defines a function to extract text from uploaded passbook or transaction images using Optical Character Recognition (OCR). The `pytesseract` library converts image-based transaction details into machine-readable text for further fraud analysis in the RAG pipeline.

In [ ]:
def extract_text_from_image(file):
    image = Image.open(file)
    text = pytesseract.image_to_string(image)
    return text

## Transaction Text Preprocessing

This cell defines a preprocessing function to clean and standardize transaction text before analysis. It converts text to lowercase, removes special characters, expands abbreviations, normalizes transaction amounts, and removes extra spaces to improve the accuracy of the RAG retrieval and LLM-based fraud detection process.

In [ ]:
import re

def preprocess(text):
    text = text.lower()

    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    text = text.replace("50k", "50000")
    text = text.replace("75k", "75000")
    text = text.replace("1lakh", "100000")
    text = text.replace("2lakh", "200000")

    text = text.replace("acc", "account")
    text = text.replace("txn", "transaction")
    text = text.replace("amt", "amount")
    text = text.replace("dr", "debit")
    text = text.replace("cr", "credit")

    text = re.sub(r'\s+', ' ', text).strip()

    return text

## Bank Statement Information Extraction

This cell defines a function to extract important transaction details from bank statement text using regular expressions. The function identifies transaction amounts, source and destination account numbers, and destination location information, which are later used for fraud analysis and explanation in the RAG-based system.

In [ ]:
import re

def parse_bank_statement(text):
    text_lower = text.lower()

    amount = re.findall(r'\b\d{4,6}\b', text_lower)
    account_numbers = re.findall(r'\b[a-z0-9]{6,}\b', text_lower)

    amount = amount[0] if amount else "Unknown"

    source_account = account_numbers[0] if len(account_numbers) > 0 else "Unknown"
    destination_account = account_numbers[1] if len(account_numbers) > 1 else "Unknown"

    # Simple keyword detection
    if "international" in text_lower or "intl" in text_lower:
        destination_location = "Foreign"
    elif "india" in text_lower:
        destination_location = "India"
    else:
        destination_location = "Unknown"

    return {
        "amount": amount,
        "source_account": source_account,
        "destination_account": destination_account,
        "destination_location": destination_location
    }

## Fraud Rule Retrieval using RAG

This cell defines a retrieval function that searches the FAISS vector database for the most relevant fraud detection rules based on the input transaction query. The retrieved rules provide contextual knowledge to the LLM, enabling more accurate and explainable fraud analysis in the RAG pipeline.

In [ ]:
def retrieve_rules(query, k=3):
    query_embedding = embed_model.encode([query])
    distances, indices = index.search(np.array(query_embedding), k)

    retrieved = [fraud_rules[i] for i in indices[0]]
    return retrieved

## Transaction Entity Extraction

This cell defines a function to extract important entities such as source name, destination name, source location, and destination location from transaction text. The extracted information helps the system generate detailed fraud explanations and improves the interpretability of the RAG and LLM-based fraud detection process.

In [ ]:
def extract_entities(text):

    parsed = parse_bank_statement(text)

    source = "Unknown"
    destination = "Unknown"
    source_location = "Unknown"
    destination_location = parsed["destination_location"]

    # Try to extract names (if present)
    if "from" in text.lower() and "to" in text.lower():
        try:
            source = text.split("from")[1].split("to")[0].strip().split()[0]
        except:
            pass

    if "to" in text.lower():
        try:
            destination = text.split("to")[1].strip()
        except:
            pass

    # fallback to account numbers
    if source == "Unknown":
        source = parsed["source_account"]

    if destination == "Unknown":
        destination = parsed["destination_account"]

    if "india" in text.lower():
        source_location = "India"

    return {
        "source_name": source,
        "destination_name": destination,
        "source_location": source_location,
        "destination_location": destination_location
    }

## Fraud Detection and Explanation using RAG and LLM

This cell defines the core fraud detection function of the system. It preprocesses transaction text, retrieves relevant fraud rules using the RAG pipeline, extracts important transaction entities, and generates fraud explanations using the Phi-2 Large Language Model. The function also performs rule-based fraud scoring, calculates confidence levels, and returns the final fraud analysis along with suggested safety steps.

In [ ]:
def detect_fraud(text):

    cleaned_text = preprocess(text)
    rules = retrieve_rules(cleaned_text)

    entities = extract_entities(text)

    prompt = f"""
Analyze this transaction and give a short answer.

Transaction:
{cleaned_text}

Entities:
Source: {entities['source_name']}
Destination: {entities['destination_name']}
Source Location: {entities['source_location']}
Destination Location: {entities['destination_location']}

Fraud Indicators:
{rules}

Instructions:
- Answer in 2–3 lines only
- Be specific to this transaction
- Do NOT give general explanations
- Do NOT list points
- Do NOT give examples

Answer:
"""

    device = "cuda" if torch.cuda.is_available() else "cpu"

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # =========================
    # 🔥 CLEAN LLM OUTPUT
    # =========================

    if "Answer:" in response:
        response = response.split("Answer:")[-1]

    if "Solution:" in response:
        response = response.split("Solution:")[-1]

    if "Follow-up" in response:
        response = response.split("Follow-up")[0]

    response = response.strip()

    # Extract only complete sentences safely
    import re
    sentences = re.findall(r'[^.]+(?:\.)', response)

    if len(sentences) > 0:
        response = " ".join(sentences[:2]).strip()
    else:
        response = "This transaction appears suspicious due to unusual transfer patterns and possible foreign involvement."

    # =========================
    # 🔥 RULE-BASED SCORING
    # =========================

    text_lower = cleaned_text.lower()

    fraud_keywords = [
        "urgent",
        "lottery",
        "prize",
        "unknown",
        "foreign",
        "international",
        "intl",
        "imps",
        "neft",
        "rtgs"
    ]

    score = sum([1 for word in fraud_keywords if word in text_lower])

    # Boost for foreign transaction
    if entities["destination_location"] == "Foreign":
        score += 2

    is_fraud = "Fraud" if score >= 2 else "Not Fraud"
    confidence = min(40 + score * 15, 95)

    # =========================
    # 🔥 FINAL OUTPUT
    # =========================

    return {
        "isFraud": is_fraud,
        "confidence": str(confidence),
        "explanation": response,
        "extract": {
            "source": entities["source_name"],
            "destination": entities["destination_name"],
            "source_location": entities["source_location"],
            "destination_location": entities["destination_location"]
        },
        "suggested_steps": [
            "Verify recipient details",
            "Contact bank if suspicious",
            "Do not proceed if unsure"
        ],
        "rule_score": score * 10
    }

## Rule-Based Fraud Scoring

This cell defines a rule-based scoring function that assigns fraud risk scores based on suspicious keywords and transaction patterns. The scoring mechanism helps estimate the fraud probability by detecting indicators such as urgent transfers, lottery scams, foreign transactions, and unusually large transaction amounts.

In [ ]:
def rule_based_score(text):
    text = text.lower()
    score = 0

    if "urgent" in text: score += 25
    if "lottery" in text or "prize" in text: score += 30
    if "unknown" in text: score += 20
    if "foreign" in text or "international" in text: score += 20
    if "large" in text or "50000" in text or "100000" in text: score += 15

    return min(score, 100)

## Processing User Input for Fraud Analysis

This cell defines the main processing function that coordinates the complete fraud detection workflow. It calls the fraud detection module, extracts transaction entities, formats the analysis results, and converts the output into plain English for user-friendly explanation and display in the application interface.

In [ ]:
def process_input(text):
    try:
        llm_output = detect_fraud(text)
        entities = extract_entities(text)

        # 🔥 Ensure llm_output is safe
        if isinstance(llm_output, dict):

            llm_output["extract"] = {
                "source": entities.get("source_name", "Unknown"),
                "destination": entities.get("destination_name", "Unknown"),
                "source_location": entities.get("source_location", "Unknown"),
                "destination_location": entities.get("destination_location", "Unknown")
            }

        result = {
            "fraud_analysis": llm_output
        }

        # 🔥 Convert safely to English
        plain_text = to_plain_english(result)

        return plain_text

    except Exception as e:
        return f"Error occurred: {str(e)}"

In [ ]:
def manual_input_fn(text):
    if not text:
        return "Please enter transaction text"
    return str(process_input(text))


def pdf_input_fn(file):
    if file is None:
        return "Please upload a PDF file"
    try:
        text = extract_text_from_pdf(file.name)
        return str(process_input(text))
    except Exception as e:
        return str(e)


def image_input_fn(file):
    if file is None:
        return "Please upload an image"
    try:
        text = extract_text_from_image(file.name)
        return str(process_input(text))
    except Exception as e:
        return str(e)


with gr.Blocks() as app:
    gr.Markdown("# Financial Fraud Detection System")
    gr.Markdown("Analyze transactions using AI (RAG + LLM)")

    with gr.Tabs():

        # Manual Input
        with gr.Tab("Manual Input"):
            txt = gr.Textbox(
                label="Enter Transaction",
                placeholder="e.g. Transfer 50000 from Ravi in India to unknown foreign account"
            )
            out = gr.Textbox(label="Result", lines=10)
            btn = gr.Button("Analyze")
            btn.click(manual_input_fn, txt, out)

        # PDF Upload
        with gr.Tab("PDF Upload"):
            pdf = gr.File(label="Upload PDF Statement")
            out_pdf = gr.Textbox(label="Result", lines=10)
            btn_pdf = gr.Button("Analyze PDF")
            btn_pdf.click(pdf_input_fn, pdf, out_pdf)

        # Image Upload
        with gr.Tab("Image Upload"):
            img = gr.File(label="Upload Passbook Image")
            out_img = gr.Textbox(label="Result", lines=10)
            btn_img = gr.Button("Analyze Image")
            btn_img.click(image_input_fn, img, out_img)

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bf21c8beee07c4dc7a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
